# Remote work salary premium: causal inference

Remote jobs often look better paid in raw salary data. The catch is composition: senior roles, US-based companies, and engineering-heavy job families are not evenly split between remote and onsite work.

This notebook treats fully remote work as a "treatment" and asks a narrower question: after adjusting for observed role and market variables, is there still a salary premium for remote jobs in this dataset?

This is not a randomized experiment. The goal is to make the comparison less naive, show the uncertainty, and be honest about what remains unobserved.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
DATA_URL = "https://raw.githubusercontent.com/YuluDuan/Hypothesis-Testing-Data-Science-salary-comparison-in-different-location/main/ds_salaries.csv"
ASSET_DIR = Path("assets")
ASSET_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["font.family"] = "DejaVu Sans"

## 1. Data and analysis sample

The dataset has data-science salary records from 2020 to 2022. I keep full-time roles and compare only the clean endpoints of the remote variable: onsite (`remote_ratio = 0`) and fully remote (`remote_ratio = 100`). Hybrid roles are real, but they blur the treatment definition for this first pass.

The outcome is log salary in USD. Effects are converted back into percent differences for the report.

In [ ]:
def job_family(title):
    title = title.lower()
    if any(word in title for word in ["manager", "lead", "head", "director", "principal"]):
        return "Lead/Management"
    if any(word in title for word in ["engineer", "architect"]):
        return "Engineering"
    if any(word in title for word in ["scientist", "research"]):
        return "Science/Research"
    if any(word in title for word in ["analyst", "analytics", "bi"]):
        return "Analytics"
    if "machine learning" in title or "ml " in title or "ai " in title:
        return "ML/AI"
    return "Other"


def load_analysis_data(url=DATA_URL):
    raw = pd.read_csv(url).drop(columns=["Unnamed: 0"], errors="ignore")
    df = raw.query("employment_type == 'FT' and remote_ratio in [0, 100]").copy()
    df = df[(df["salary_in_usd"] >= 10_000) & (df["salary_in_usd"] <= 450_000)].copy()
    df["remote"] = (df["remote_ratio"] == 100).astype(int)
    df["log_salary"] = np.log(df["salary_in_usd"])
    df["job_family"] = df["job_title"].map(job_family)
    df["employee_market"] = np.where(df["employee_residence"] == "US", "US", "Non-US")
    df["company_market"] = np.where(df["company_location"] == "US", "US", "Non-US")
    return raw, df

raw, df = load_analysis_data()
print(f"Raw rows: {len(raw):,}")
print(f"Analysis rows: {len(df):,}")
print(df["remote"].value_counts().rename({0: "Onsite", 1: "Remote"}))
df.head()

## 2. Identification setup

The treatment is fully remote work. The adjustment set is deliberately observable and business-readable:

- work year
- seniority
- job family
- employee market: US vs non-US
- company market: US vs non-US
- company size

The assumption is conditional exchangeability: after these variables, remote and onsite roles are comparable enough for an adjusted contrast. That is a strong assumption. It does not cover skills, company brand, negotiation, equity, benefits, or exact city.

In [ ]:
FEATURES = ["work_year", "experience_level", "job_family", "employee_market", "company_market", "company_size"]
CAT_FEATURES = [c for c in FEATURES if c != "work_year"]


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
            ("num", "passthrough", ["work_year"]),
        ],
        verbose_feature_names_out=False,
    )

preprocessor = make_preprocessor()
X = preprocessor.fit_transform(df[FEATURES])
y = df["log_salary"].to_numpy()
t = df["remote"].to_numpy()

propensity_model = LogisticRegression(max_iter=2_000, C=1.0)
propensity_model.fit(X, t)
propensity = np.clip(propensity_model.predict_proba(X)[:, 1], 0.03, 0.97)

print(f"Propensity AUC: {roc_auc_score(t, propensity):.3f}")
print(f"Propensity range: {propensity.min():.3f} to {propensity.max():.3f}")

## 3. Estimators

I compare five estimates:

- naive difference in mean log salary
- regression adjustment
- inverse probability weighting (IPW)
- doubly robust / AIPW estimate
- nearest-neighbor matching on the propensity score

The report uses the doubly robust estimate as the headline because it combines an outcome model with treatment weights. If either side is reasonably specified, the estimate is less fragile than a single adjustment.

In [ ]:
def fit_nuisance_models(X, y, t, seed=RANDOM_STATE):
    m0 = GradientBoostingRegressor(random_state=seed, max_depth=2, n_estimators=80, learning_rate=0.05)
    m1 = GradientBoostingRegressor(random_state=seed, max_depth=2, n_estimators=80, learning_rate=0.05)
    m0.fit(X[t == 0], y[t == 0])
    m1.fit(X[t == 1], y[t == 1])
    return m0, m1


def estimate_effects(data, seed=RANDOM_STATE):
    local_preprocessor = make_preprocessor()
    local_X = local_preprocessor.fit_transform(data[FEATURES])
    y = data["log_salary"].to_numpy()
    t = data["remote"].to_numpy()

    p_model = LogisticRegression(max_iter=2_000, C=1.0).fit(local_X, t)
    e = np.clip(p_model.predict_proba(local_X)[:, 1], 0.03, 0.97)
    m0, m1 = fit_nuisance_models(local_X, y, t, seed=seed)
    mu0 = m0.predict(local_X)
    mu1 = m1.predict(local_X)

    naive = y[t == 1].mean() - y[t == 0].mean()
    regression_adjustment = np.mean(mu1 - mu0)
    ipw = np.mean(t * y / e - (1 - t) * y / (1 - e))
    doubly_robust = np.mean(mu1 - mu0 + t * (y - mu1) / e - (1 - t) * (y - mu0) / (1 - e))

    nn = NearestNeighbors(n_neighbors=1).fit(e[t == 0].reshape(-1, 1))
    _, idx = nn.kneighbors(e[t == 1].reshape(-1, 1))
    matching = y[t == 1].mean() - y[t == 0][idx[:, 0]].mean()

    return {
        "naive": naive,
        "regression_adjustment": regression_adjustment,
        "ipw": ipw,
        "doubly_robust": doubly_robust,
        "matching": matching,
        "propensity": e,
        "mu0": mu0,
        "mu1": mu1,
        "propensity_auc": roc_auc_score(t, e),
    }

estimates = estimate_effects(df)
summary = pd.DataFrame([
    {"Estimator": key.replace("_", " ").title(), "Log effect": value, "Percent effect": (np.exp(value) - 1) * 100}
    for key, value in estimates.items()
    if key in ["naive", "regression_adjustment", "ipw", "doubly_robust", "matching"]
])
summary

## 4. Bootstrap uncertainty

The dataset is small, so I use a simple row bootstrap. Each resample refits the propensity model and the outcome models. That is slower than resampling residuals, but it reflects the modeling uncertainty better.

In [ ]:
def bootstrap_effects(df, n_boot=300, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    rows = []
    n = len(df)
    for i in range(n_boot):
        sample = df.iloc[rng.integers(0, n, n)].copy()
        if sample["remote"].nunique() < 2:
            continue
        try:
            est = estimate_effects(sample, seed=seed + i)
        except Exception:
            continue
        rows.append({k: est[k] for k in ["naive", "regression_adjustment", "ipw", "doubly_robust", "matching"]})
    return pd.DataFrame(rows)

boot = bootstrap_effects(df)
ci_rows = []
for estimator in ["naive", "regression_adjustment", "ipw", "doubly_robust", "matching"]:
    point = estimates[estimator]
    low, high = np.quantile(boot[estimator], [0.025, 0.975])
    ci_rows.append({
        "Estimator": estimator.replace("_", " ").title(),
        "Point": (np.exp(point) - 1) * 100,
        "CI low": (np.exp(low) - 1) * 100,
        "CI high": (np.exp(high) - 1) * 100,
    })
ci = pd.DataFrame(ci_rows)
ci

## 5. Diagnostics and charts

The report needs to show more than the final number. The key diagnostics are overlap and balance. If remote and onsite roles do not overlap on observed covariates, no estimator saves the analysis.

In [ ]:
def percent(x):
    return (np.exp(x) - 1) * 100

plot_ci = ci.copy()
order = ["Naive", "Regression Adjustment", "Ipw", "Doubly Robust", "Matching"]
plot_ci["Estimator"] = pd.Categorical(plot_ci["Estimator"], order, ordered=True)
plot_ci = plot_ci.sort_values("Estimator")

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(plot_ci))
ax.errorbar(
    plot_ci["Point"],
    y_pos,
    xerr=[plot_ci["Point"] - plot_ci["CI low"], plot_ci["CI high"] - plot_ci["Point"]],
    fmt="o",
    color="#111827",
    ecolor="#6b7280",
    elinewidth=2,
    capsize=4,
)
ax.axvline(0, color="#dc2626", linestyle="--", linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_ci["Estimator"])
ax.set_xlabel("Estimated remote salary effect (%)")
ax.set_title("Remote salary premium shrinks after adjustment")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(ASSET_DIR / "01_effect_estimates.png", bbox_inches="tight")
plt.show()

In [ ]:
plot_df = df.assign(propensity=estimates["propensity"], Work_Mode=np.where(df["remote"] == 1, "Remote", "Onsite"))
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=plot_df, x="propensity", hue="Work_Mode", bins=20, stat="density", common_norm=False, alpha=0.45, ax=ax)
ax.set_title("Propensity overlap: remote roles are common across most profiles")
ax.set_xlabel("Estimated probability of being fully remote")
fig.tight_layout()
fig.savefig(ASSET_DIR / "02_propensity_overlap.png", bbox_inches="tight")
plt.show()

In [ ]:
feature_names = preprocessor.get_feature_names_out()
X_df = pd.DataFrame(X, columns=feature_names)
weights = np.where(t == 1, 1 / estimates["propensity"], 1 / (1 - estimates["propensity"]))


def smd(values, treatment, weights=None):
    values = np.asarray(values, dtype=float)
    treatment = np.asarray(treatment)
    if weights is None:
        m1, m0 = values[treatment == 1].mean(), values[treatment == 0].mean()
        v1, v0 = values[treatment == 1].var(), values[treatment == 0].var()
    else:
        w1, w0 = weights[treatment == 1], weights[treatment == 0]
        x1, x0 = values[treatment == 1], values[treatment == 0]
        m1, m0 = np.average(x1, weights=w1), np.average(x0, weights=w0)
        v1 = np.average((x1 - m1) ** 2, weights=w1)
        v0 = np.average((x0 - m0) ** 2, weights=w0)
    pooled = np.sqrt((v1 + v0) / 2)
    return 0 if pooled == 0 else (m1 - m0) / pooled

balance = pd.DataFrame({
    "feature": feature_names,
    "Before weighting": [abs(smd(X_df[c], t)) for c in feature_names],
    "After IPW": [abs(smd(X_df[c], t, weights=weights)) for c in feature_names],
})
balance_long = balance.sort_values("Before weighting", ascending=False).head(12).melt("feature", var_name="Stage", value_name="Absolute SMD")

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=balance_long, y="feature", x="Absolute SMD", hue="Stage", ax=ax, palette=["#ef4444", "#2563eb"])
ax.axvline(0.1, color="#111827", linestyle="--", linewidth=1)
ax.set_title("IPW improves balance on the largest observed differences")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(ASSET_DIR / "03_covariate_balance.png", bbox_inches="tight")
plt.show()

In [ ]:
heterogeneity = []
working = df.assign(e=estimates["propensity"], mu0=estimates["mu0"], mu1=estimates["mu1"])
for level, sub in working.groupby("experience_level"):
    if sub["remote"].nunique() < 2 or len(sub) < 30:
        continue
    yy = sub["log_salary"].to_numpy()
    tt = sub["remote"].to_numpy()
    ee = sub["e"].to_numpy()
    mu0 = sub["mu0"].to_numpy()
    mu1 = sub["mu1"].to_numpy()
    effect = np.mean(mu1 - mu0 + tt * (yy - mu1) / ee - (1 - tt) * (yy - mu0) / (1 - ee))
    heterogeneity.append({"Experience": level, "Rows": len(sub), "Remote effect (%)": percent(effect)})
heterogeneity = pd.DataFrame(heterogeneity)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=heterogeneity, x="Experience", y="Remote effect (%)", ax=ax, color="#2563eb")
ax.axhline(0, color="#dc2626", linestyle="--", linewidth=1)
ax.set_title("The adjusted effect is not uniform by seniority")
fig.tight_layout()
fig.savefig(ASSET_DIR / "04_effect_by_seniority.png", bbox_inches="tight")
plt.show()
heterogeneity

In [ ]:
dist_df = df.assign(Work_Mode=np.where(df["remote"] == 1, "Remote", "Onsite"))
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(data=dist_df, x="salary_in_usd", hue="Work_Mode", common_norm=False, fill=True, alpha=0.35, ax=ax)
ax.set_xlim(0, 350_000)
ax.set_xlabel("Salary in USD")
ax.set_title("Raw salary distributions before adjustment")
fig.tight_layout()
fig.savefig(ASSET_DIR / "05_raw_salary_distribution.png", bbox_inches="tight")
plt.show()

## 6. Readout

The raw comparison says remote roles earn more. After adjustment, the headline estimate is smaller and less dramatic. That is the point of the project.

A good LinkedIn version should lead with the contrast between the naive estimate and the doubly robust estimate. The caveat belongs in the same paragraph: this is observational salary data, not a randomized remote-work experiment.

In [ ]:
result = {
    "raw_rows": int(len(raw)),
    "analysis_rows": int(len(df)),
    "remote_rows": int(df["remote"].sum()),
    "onsite_rows": int((1 - df["remote"]).sum()),
    "propensity_auc": float(estimates["propensity_auc"]),
    "naive_pct": float(percent(estimates["naive"])),
    "doubly_robust_pct": float(percent(estimates["doubly_robust"])),
    "doubly_robust_ci_low": float(ci.loc[ci["Estimator"] == "Doubly Robust", "CI low"].iloc[0]),
    "doubly_robust_ci_high": float(ci.loc[ci["Estimator"] == "Doubly Robust", "CI high"].iloc[0]),
    "ipw_pct": float(percent(estimates["ipw"])),
    "matching_pct": float(percent(estimates["matching"])),
}
Path("results_summary.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
result